<a href="https://colab.research.google.com/github/deltaVOCALOID09378/DiffSinger_Edit_By_DELTA_SYNTH/blob/main/DiffSinger_colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# -*- coding: utf-8 -*-
# ==============================================================================
# Date: 23/10/2026 (B.E. 2569) / 23 October 2026
# File Name: Program.py
# Version: 1.0
# Location: /content/Program.py
# Function: DiffSinger Data Processing & Training Configuration Pipeline
# Credit: Made And Checked By DELTA SYNTH & All Code Agentic AI Engine
#         [ Original by Kei / MLo7 ]
# Description: สคริปต์จัดการข้อมูลและตั้งค่าการฝึกสอนโมเดล DiffSinger (Acoustic & Variance)
#              รองรับการจัดเก็บข้อมูลบน Google Drive อย่างเป็นระบบ (raw, export, binary)
#              พร้อมการสร้างไฟล์ Config อัตโนมัติเป็นภาษาไทยที่เข้าใจง่าย
# ==============================================================================

import os
import sys
import yaml
import json
import shutil
import logging
from pathlib import Path

# ------------------------------------------------------------------------------
# การตั้งค่า Logger (ระบบบันทึกสถานะการทำงาน)
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("Delta_DiffSinger")


class DiffSingerWorkspace:
    """
    คลาสสำหรับจัดการสภาพแวดล้อมและข้อมูลของ DiffSinger บน Google Drive
    """
    def __init__(self, model_name: str, drive_base_path: str = "/content/drive/MyDrive/My_model_file"):
        self.model_name = model_name
        self.base_dir = Path(drive_base_path) / self.model_name
        
        # กำหนดเส้นทางโฟลเดอร์ตามมาตรฐานที่ท่านเดลต้าระบุ
        self.raw_dir = self.base_dir / "raw"
        self.export_dir = self.base_dir / "export"
        self.binary_dir = self.base_dir / "binary"
        
        self.config_acoustic_dir = self.base_dir / "Acoustics_Config"
        self.config_variance_dir = self.base_dir / "Variance_Config"

    def mount_google_drive(self):
        """
        เชื่อมต่อและขอสิทธิ์เข้าถึง Google Drive
        """
        logger.info("กำลังตรวจสอบการเชื่อมต่อ Google Drive...")
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount("/content/drive")
                logger.info("เชื่อมต่อ Google Drive สำเร็จเรียบร้อยแล้วครับ")
            else:
                logger.info("Google Drive ถูกเชื่อมต่อไว้แล้วครับ")
        except ImportError:
            logger.warning("ไม่พบโมดูล google.colab ข้ามขั้นตอนการเชื่อมต่อ Drive (อาจรันอยู่บน Local)")

    def setup_directories(self):
        """
        สร้างโครงสร้างโฟลเดอร์สำหรับจัดเก็บข้อมูลโมเดล
        """
        logger.info(f"กำลังจัดเตรียมพื้นที่ทำงานสำหรับโมเดล: {self.model_name}")
        directories = [
            self.raw_dir,
            self.export_dir,
            self.binary_dir,
            self.config_acoustic_dir,
            self.config_variance_dir
        ]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
            logger.info(f"เตรียมโฟลเดอร์พร้อมใช้งาน: {directory}")

    def generate_configs(self, num_spk: int = 1, use_gpu: bool = True):
        """
        สร้างไฟล์ Acoustics.yaml และ Variance.yaml อย่างเป็นทางการ
        """
        logger.info("กำลังสร้างไฟล์ตั้งค่า (Configuration Files) สำหรับการฝึกสอน...")
        
        # 1. การตั้งค่าพื้นฐานสำหรับ Acoustics
        acoustic_config = {
            "model_type": "acoustic",
            "num_spk": num_spk,
            "use_spk_id": num_spk > 1,
            "raw_data_dir": str(self.raw_dir),
            "binary_data_dir": str(self.binary_dir),
            "export_dir": str(self.export_dir),
            "pl_trainer_accelerator": "gpu" if use_gpu else "cpu",
            "pl_trainer_precision": "16-mixed",
            "hidden_size": 256,
            "main_loss_type": "l2",
            "pe": "rmvpe",
            "augmentation_args": {
                "random_pitch_shifting": {"enabled": True},
                "random_time_stretching": {"enabled": True}
            },
            "note": "การตั้งค่าโมเดล Acoustic อัตโนมัติ โดย DELTA SYNTH"
        }
        
        # 2. การตั้งค่าพื้นฐานสำหรับ Variance
        variance_config = {
            "model_type": "variance",
            "num_spk": num_spk,
            "use_spk_id": num_spk > 1,
            "raw_data_dir": str(self.raw_dir),
            "binary_data_dir": str(self.binary_dir),
            "export_dir": str(self.export_dir),
            "pl_trainer_accelerator": "gpu" if use_gpu else "cpu",
            "pl_trainer_precision": "16-mixed",
            "predict_dur": True,
            "predict_pitch": True,
            "predict_energy": False,
            "predict_breathiness": False,
            "pe": "rmvpe",
            "note": "การตั้งค่าโมเดล Variance อัตโนมัติ โดย DELTA SYNTH"
        }

        # บันทึกไฟล์ Acoustics.yaml
        acoustic_path = self.config_acoustic_dir / "Acoustics.yaml"
        with open(acoustic_path, "w", encoding="utf-8") as f:
            yaml.dump(acoustic_config, f, allow_unicode=True, sort_keys=False)
        logger.info(f"บันทึกไฟล์การตั้งค่า Acoustic สำเร็จที่: {acoustic_path}")

        # บันทึกไฟล์ Variance.yaml
        variance_path = self.config_variance_dir / "Variance.yaml"
        with open(variance_path, "w", encoding="utf-8") as f:
            yaml.dump(variance_config, f, allow_unicode=True, sort_keys=False)
        logger.info(f"บันทึกไฟล์การตั้งค่า Variance สำเร็จที่: {variance_path}")

    def execute_pipeline(self):
        """
        ฟังก์ชันหลักในการรันกระบวนการทั้งหมด
        """
        try:
            print("=" * 70)
            print(f"🌟 เริ่มต้นระบบจัดการข้อมูล DiffSinger - โมเดล: {self.model_name}")
            print("=" * 70)
            
            self.mount_google_drive()
            self.setup_directories()
            self.generate_configs()
            
            print("=" * 70)
            print("✅ กระบวนการเตรียมข้อมูลและการตั้งค่าเสร็จสมบูรณ์เรียบร้อยแล้วครับ")
            print("ท่านสามารถนำข้อมูลไฟล์เสียงมาใส่ในโฟลเดอร์ 'raw' และเริ่มการฝึกสอนได้ทันที")
            print("=" * 70)
            
        except Exception as e:
            logger.error(f"❌ พบข้อผิดพลาดระหว่างดำเนินการ: {str(e)}")
            logger.error(traceback.format_exc())


if __name__ == "__main__":
    # ==============================================================================
    # ส่วนสำหรับผู้ใช้งาน (User Settings)
    # ==============================================================================
    
    # 1. กรุณาระบุชื่อโมเดลที่ต้องการสร้างหรือทำงานด้วย
    MODEL_NAME = "Miku_Thai_VCCV"
    
    # 2. เริ่มต้นระบบ
    workspace = DiffSingerWorkspace(model_name=MODEL_NAME)
    workspace.execute_pipeline()